# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print a summary
print(f"Dataset name: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields (columns), and their `@id`s.

In [ ]:
# List all record sets and their schema, including fields and columns referenced by @id
print("Available record sets in this dataset:")
for record_set in metadata.record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    # List fields
    print("  Fields (columns):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all record sets and store them in DataFrames, using their @id as the key
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    # Records are iterables (generators)
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    # Show first 3 columns if any data
    if not df.empty:
        print(f"Columns for {rs_id}: {df.columns.tolist()[:3]} ... (total {len(df.columns)})")
    else:
        print(f"No records loaded for {rs_id}")
print(f"\nDataFrames loaded: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes example operations for filtering, normalizing, and grouping using `@id` references only.

In [ ]:
# For demonstration, we find the first record set with data and a numeric field
numeric_types = {"Number", "Float", "Integer", "Double"}

selected_rs_id = None
selected_numeric_field_id = None
selected_group_field_id = None

for rs in metadata.record_sets:
    df = dataframes.get(rs.id, pd.DataFrame())
    if not df.empty:
        # Find a numeric field by type
        for field in rs.fields:
            if field.data_type in numeric_types and field.id in df.columns:
                selected_rs_id = rs.id
                selected_numeric_field_id = field.id
                break
        # Find a non-numeric field for grouping
        for field in rs.fields:
            if field.data_type == "Text" and field.id in df.columns:
                selected_group_field_id = field.id
                break
        if selected_rs_id:
            break

if selected_rs_id and selected_numeric_field_id:
    print(f"Using record set: {selected_rs_id}")
    print(f"Numeric field: {selected_numeric_field_id}")
    print(f"Group-by field: {selected_group_field_id}")

    df = dataframes[selected_rs_id]
    # Ensure numeric field is numeric
    df[selected_numeric_field_id] = pd.to_numeric(df[selected_numeric_field_id], errors="coerce")
    # Drop rows with NaN in the numeric field
    df = df.dropna(subset=[selected_numeric_field_id])
    
    # Example EDA: filter outliers, normalize, group
    field_mean = df[selected_numeric_field_id].mean()
    field_std = df[selected_numeric_field_id].std()
    threshold = field_mean + field_std
    filtered_df = df[df[selected_numeric_field_id] > threshold]
    print(f"Filtered records ({selected_numeric_field_id} > mean+std): {len(filtered_df)} rows")

    norm_col_name = selected_numeric_field_id + "_normalized"
    filtered_df[norm_col_name] = (filtered_df[selected_numeric_field_id] - field_mean) / field_std
    print(f"Showing normalized values of {selected_numeric_field_id}:")
    print(filtered_df[[selected_numeric_field_id, norm_col_name]].head())

    if selected_group_field_id and selected_group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(selected_group_field_id)[selected_numeric_field_id].mean().reset_index()
        print(f"Mean {selected_numeric_field_id} by {selected_group_field_id} (grouped):")
        print(grouped_df.head())
else:
    print("No suitable record set or numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize distribution of the selected numeric field
if selected_rs_id and selected_numeric_field_id:
    df = dataframes[selected_rs_id]
    df[selected_numeric_field_id] = pd.to_numeric(df[selected_numeric_field_id], errors='coerce')
    plt.figure(figsize=(7,4))
    df[selected_numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {selected_numeric_field_id} in record set {selected_rs_id}")
    plt.xlabel(selected_numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, examine, and analyze the FAIR² dataset using the `mlcroissant` library. By referencing all entities strictly by their `@id`, we maintained reproducibility and machine-actionability. This workflow can be adapted for similar Croissant-structured datasets for rigorous exploratory analysis.

Key steps included loading metadata, enumerating record sets and fields, extracting actual data, conducting basic EDA (including filtering and normalization), and producing visual summaries for measurable features. Continue to adapt and extend this notebook for deeper analyses as required.